<a href="https://colab.research.google.com/github/iqra-amer/Flyrank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 1. My Lane as an ML Task

Lane: Refresh / Content Opportunity Scoring

ML Task Type: Scoring

Why: The goal is to assign a priority score to each webpage based on multiple observable signals such as traffic trend, impressions, content age, and engagement. The scores can then be sorted to produce a ranked review queue for content managers.

In [8]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

print("Repository is ready!")

Repository is ready!


In [9]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)

df.head()


Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 2. Target or Proxy

The ideal target is a **Refresh Priority Score** assigned to each webpage.

The dataset does not contain such a column directly. Instead, this score would be estimated using several observable signals, including:

- Trend direction
- Impressions
- Average search position
- Click-through rate (CTR)
- Content age
- Engagement

These features act as proxies that together help estimate which webpages should be reviewed first.

In [10]:
# Create a small sample

target_example = df[[
    "content_id",
    "trend_direction",
    "impressions_90d",
    "content_age_days"
]].head().copy()

# Example target column
target_example["refresh_priority_score"] = [
    "High",
    "Medium",
    "High",
    "Low",
    "Medium"
]

target_example

,content_id,trend_direction,impressions_90d,content_age_days,refresh_priority_score
0,content_304f48230142,down,3803,187,High
1,content_a1fb4e703a9e,down,15320,445,Medium
2,content_9aa793d4d895,down,12581,141,High
3,content_331d6c4de07b,stable,11751,463,Low
4,content_d99b7a2d90ca,down,19140,263,Medium


# 3. Success Metric

Success means that the scoring system consistently places the webpages most likely to benefit from review near the top of the priority list.

This allows content managers to spend their time reviewing the highest-value opportunities first.

The goal is to improve decision making rather than replace human judgment.

In [11]:
print("Total webpages:", len(df))

print("\nTrend Direction")
display(df["trend_direction"].value_counts())

print("\nAge Tier")
display(df["age_tier"].value_counts())

print("\nImpression Statistics")
display(df["impressions_90d"].describe())

Total webpages: 30000

Trend Direction


,count
trend_direction,
down,16262
stable,5962
up,4388
new,2236
flat,1152



Age Tier


,count
age_tier,
91-180,11780
181-365,11368
365+,6360
31-90,492



Impression Statistics


,impressions_90d
count,30000.000000
mean,5200.366300
std,16838.019547
min,1.000000
25%,81.000000
50%,731.000000
75%,3615.250000
max,517715.000000


# 4. Unit of Analysis

The unit of analysis is **one webpage**.

Each row in the dataset represents one webpage together with its search performance, engagement metrics, freshness information, and content characteristics.

The ML system assigns one refresh priority score to each webpage.

In [12]:
unit_of_analysis = df[[
    "content_id",
    "client_id",
    "trend_direction",
    "impressions_90d",
    "ctr",
    "avg_position",
    "content_age_days"
]].head()

unit_of_analysis

,content_id,client_id,trend_direction,impressions_90d,ctr,avg_position,content_age_days
0,content_304f48230142,client_f369cb89fc,down,3803,0.76,10.6,187
1,content_a1fb4e703a9e,client_4e07408562,down,15320,0.05,20.3,445
2,content_9aa793d4d895,client_7f2253d7e2,down,12581,0.09,36.5,141
3,content_331d6c4de07b,client_19581e27de,stable,11751,0.49,6.2,463
4,content_d99b7a2d90ca,client_3fdba35f04,down,19140,0.13,44.0,263


# 5. Why ML Beats a Fixed Rule

A fixed rule would treat all pages with the same condition equally. For example, a rule such as "refresh every page with a down trend" ignores other important information like impressions, content age, click-through rate, engagement, and search position.

Machine learning can combine multiple signals to estimate the overall priority of each webpage. This produces a more balanced and consistent scoring system that helps content managers focus on the pages where a review is likely to have the greatest impact.

The goal is not to replace human judgment but to support better decision-making using data.

# 6. Self-Check

- [x] Selected the ML task type.
- [x] Defined the target/proxy.
- [x] Defined a success metric.
- [x] Showed the unit of analysis.
- [x] Sketched the target column.
- [x] Explained why ML is better than a fixed rule.
- [x] Connected the output to a real business action.